# Sanity check — in-domain AUC on every attached dataset

What this does
--------------
For each dataset in `DATASETS_TO_TEST`, runs one
`raw × logreg × source-only × seed=0` config and prints the in-domain AUC
alongside the expected literature value.

* **Green check (✓)** — loader, Kaggle slug, and pipeline all work.
* **Yellow warning (⚠)** — AUC is outside the literature band; the loader
  may be parsing labels or channels wrong.  Check the loader before
  scaling up.
* **Red cross (✗)** — the path does not resolve.  Edit `KAGGLE_PATHS` in
  cell 2 to match the actual slug Kaggle attached.

This notebook is the **first step**.  Once every dataset you care about
shows a ✓, switch to `run_experiments.ipynb` and crank `PRIORITY = 1`.

Setup before running
--------------------
1. Repo is at `/kaggle/working/cross-domain-experiment/` (upload or git clone).
2. Attach the relevant Kaggle datasets via **+Add data** (start with one).
3. Edit `DATASETS_TO_TEST` in cell 2 to match what you attached.
4. Run all cells top to bottom — total wall time is 1–5 min per dataset.

In [ ]:
# ── 1. Environment + Kaggle slug paths ──────────────────────────────────────
import os, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/cross-domain-experiment')
if not REPO_DIR.exists():
    raise SystemExit(
        f'Repo not at {REPO_DIR}.  Upload it via +Add data, or run\n'
        '  !git clone https://github.com/PeregrinAl/cross-domain-experiment.git '
        f'{REPO_DIR}'
    )

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))

os.environ.setdefault('DATA_ROOT',   '/kaggle/input')
os.environ.setdefault('OUTPUT_ROOT', '/kaggle/working')

# Kaggle slug paths — EDIT to match what Kaggle attached.
# After +Add data, click the dataset name on the right sidebar and
# the path is shown at the top of the file listing.  Common variants:
#   /kaggle/input/<slug>
#   /kaggle/input/<slug>/<extracted_subdir>
KAGGLE_PATHS = {
    'mitbih':    '/kaggle/input/mitbih-database/mitbih_database',
    'sleep-edf': '/kaggle/input/sleepedf-78/sleep-edf-database-expanded/sleep-cassette',
    'cwru':      '/kaggle/input/cwru-data',
    'chbmit':    '/kaggle/input/seizure-eeg-chb-mit-and-siena-scalp',
    'mimii':     '/kaggle/input/mimii-pump-sound-dataset',
}

# Which datasets to sanity-check this run.  Start with ONE — get a ✓ before
# attaching the next one.  Order doesn't matter; each runs independently.
DATASETS_TO_TEST = ['cwru']
# Examples once cwru works:
#   DATASETS_TO_TEST = ['cwru', 'chbmit']
#   DATASETS_TO_TEST = ['mitbih', 'sleep-edf', 'cwru', 'chbmit', 'mimii']

# Wire slug paths into the env vars run_one_config._register_dataset reads.
_ENV_VARS = {
    'mitbih':    'KAGGLE_MITBIH_DIR',
    'sleep-edf': 'KAGGLE_SLEEP_EDF_DIR',
    'cwru':      'KAGGLE_CWRU_DIR',
    'chbmit':    'KAGGLE_CHBMIT_DIR',
    'mimii':     'KAGGLE_MIMII_DIR',
}
for ds in DATASETS_TO_TEST:
    os.environ[_ENV_VARS[ds]] = KAGGLE_PATHS[ds]

# senaca/mimii-pump-sound-dataset ships only the pump machine.  If you've
# uploaded valve/fan/slider separately, change this here.
os.environ['KAGGLE_MIMII_MACHINE'] = 'pump'

print(f'Repo at        : {REPO_DIR}')
print(f'Datasets to test: {DATASETS_TO_TEST}')
for ds in DATASETS_TO_TEST:
    print(f'  {ds:10s} → {KAGGLE_PATHS[ds]}')

In [ ]:
# ── 2. Install runtime deps not in Kaggle's base image ──────────────────────
import subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'mne', 'wfdb', 'scikit-posthocs',
])

In [ ]:
# ── 3. Verify each attached dataset path resolves ───────────────────────────
print('Checking dataset paths …\n')
missing = []
for ds in DATASETS_TO_TEST:
    path = Path(KAGGLE_PATHS[ds])
    if not path.exists():
        print(f'  ✗ {ds:10s} NOT FOUND at {path}')
        missing.append(ds)
        continue
    # Sample a few files so you can eyeball the layout
    sample = next(iter(path.rglob('*.*')), None)
    # Quick count — capped at 100 so this stays instant on huge trees
    n = 0
    for _ in path.rglob('*'):
        n += 1
        if n > 99:
            n = '100+'
            break
    print(f'  ✓ {ds:10s} {n} files under {path}')
    if sample is not None:
        print(f'              e.g. {sample.relative_to(path)}')

if missing:
    raise SystemExit(
        f'\nFix slug paths in cell 1 for: {missing}\n'
        'On Kaggle, click the dataset in the right sidebar and copy the '
        'path shown above the file listing.'
    )

In [ ]:
# ── 4. Run one config per dataset ───────────────────────────────────────────
import time
from pathlib import Path

OUTPUT_CSV = Path('/kaggle/working/sanity_results.csv')
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()  # fresh run, no stale rows

LOG_FILE = Path('/kaggle/working/sanity.log')
if LOG_FILE.exists():
    LOG_FILE.unlink()

for ds in DATASETS_TO_TEST:
    print(f'\n── {ds} ─────────────────────────────────────────')
    cmd = [
        sys.executable, str(REPO_DIR / 'run_one_config.py'),
        '--dataset',        ds,
        '--representation', 'raw',
        '--model',          'logreg',
        '--da-method',      'source-only',
        '--seed',           '0',
        '--output-csv',     str(OUTPUT_CSV),
        '--log-level',      'INFO',
    ]
    t0 = time.time()
    with open(LOG_FILE, 'a') as lf:
        lf.write(f'\n=== {ds} ===\n')
        proc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, text=True)
    dt = time.time() - t0
    if proc.returncode != 0:
        print(f'  ✗ FAILED ({dt:.1f}s)  — see /kaggle/working/sanity.log tail below')
        print(LOG_FILE.read_text().splitlines()[-30:])
    else:
        print(f'  ✓ done ({dt:.1f}s)')

In [ ]:
# ── 5. Summary: in-domain AUC vs literature expectation ─────────────────────
import csv

# Literature in-domain AUC bands (source-only logreg on raw windows).
# These are coarse: anything inside the band means the loader is sane;
# anything wildly outside means something is parsed wrong.
EXPECTED = {
    'mitbih':    (0.85, 0.99),
    'sleep-edf': (0.90, 1.00),
    'cwru':      (0.95, 1.00),   # README explicitly documents ~0.97-0.99
    'chbmit':    (0.85, 0.99),
    'mimii':     (0.70, 0.90),
}

if not OUTPUT_CSV.exists():
    raise SystemExit('No CSV written — every run failed in cell 4.')

print(f'{"dataset":12s} {"in_dom":>8s} {"target":>8s} {"Δ":>7s} {"expected band":>16s}  status')
print('-' * 72)
with open(OUTPUT_CSV) as f:
    for row in csv.DictReader(f):
        ds = row['dataset']
        in_auc  = float(row['in_domain_auc'])
        tgt_auc = float(row['cross_domain_auc'])
        delta   = float(row['delta_auc'])
        lo, hi = EXPECTED.get(ds, (0.5, 1.0))
        mark = '✓' if lo <= in_auc <= hi else '⚠'
        print(
            f'{ds:12s} {in_auc:8.4f} {tgt_auc:8.4f} {delta:+7.4f}  '
            f'{lo:5.2f} … {hi:5.2f}     {mark}'
        )

print()
print('✓ inside band → loader OK, proceed to run_experiments.ipynb')
print('⚠ outside band → check loader for that dataset before scaling up')
print('Full per-config log: /kaggle/working/sanity.log')